# 02 – Data Cleaning: Rejected Applications

**Goal:** Clean Lending Club's rejected loan applications (2007–2018) and load them into PostgreSQL,
so they can be joined with approved loans to analyze approval rates.

## 1. Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import io
import os
import time
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

RAW_DIR = Path("../data/raw")
REJECTED_FILE = RAW_DIR / "rejected_2007_to_2018Q4.csv"

print(REJECTED_FILE.exists())

True


## 2. First look
The file has ~27.6M rows, so it is explored with a 5-row preview and a 1M-row sample first.
The sample is **not random** (the file is not sorted by date), so it is used to check formats, not to compute rates.

In [2]:
preview = pd.read_csv(REJECTED_FILE, nrows=5)
print(preview.shape)
preview

(5, 9)


,Amount Requested,Application Date,Loan Title,Risk_Score,Debt-To-Income Ratio,Zip Code,State,Employment Length,Policy Code
0,"1,000.00",2007-05-26,Wedding Covered but No Honeymoon,693.00,10%,481xx,NM,4 years,0.00
1,"1,000.00",2007-05-26,Consolidating Debt,703.00,10%,010xx,MA,< 1 year,0.00
2,"11,000.00",2007-05-27,Want to consolidate my debt,715.00,10%,212xx,MD,1 year,0.00
3,"6,000.00",2007-05-27,waksman,698.00,38.64%,017xx,MA,< 1 year,0.00
4,"1,500.00",2007-05-27,mdrigo,509.00,9.43%,209xx,MD,< 1 year,0.00


In [3]:
sample = pd.read_csv(REJECTED_FILE, nrows=1_000_000, low_memory=False)
print(sample.shape)
sample.info(memory_usage="deep")

(1000000, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   Amount Requested      1000000 non-null  float64
 1   Application Date      1000000 non-null  str    
 2   Loan Title            999972 non-null   str    
 3   Risk_Score            773401 non-null   float64
 4   Debt-To-Income Ratio  1000000 non-null  str    
 5   Zip Code              999958 non-null   str    
 6   State                 999979 non-null   str    
 7   Employment Length     980283 non-null   str    
 8   Policy Code           1000000 non-null  float64
dtypes: float64(3), str(6)
memory usage: 110.2 MB


In [4]:
sample.isna().sum()

Amount Requested             0
Application Date             0
Loan Title                  28
Risk_Score              226599
Debt-To-Income Ratio         0
Zip Code                    42
State                       21
Employment Length        19717
Policy Code                  0
dtype: int64

In [5]:
for column in sample.columns:
    print(f"--- {column} ({sample[column].dtype}), unique: {sample[column].nunique():,}")
    print(sample[column].value_counts(dropna=False).head(5))
    print()

--- Amount Requested (float64), unique: 2,026
Amount Requested
10,000.00    107747
5,000.00     104246
15,000.00     71183
20,000.00     70303
35,000.00     69145
Name: count, dtype: int64

--- Application Date (str), unique: 2,066
Application Date
2016-04-12    20145
2016-04-11    19978
2016-04-05    17229
2016-04-13    15569
2016-04-19    15282
Name: count, dtype: int64

--- Loan Title (str), unique: 61,364
Loan Title
debt_consolidation    356889
other                 124362
credit_card            80939
home_improvement       55745
car                    51506
Name: count, dtype: int64

--- Risk_Score (float64), unique: 534
Risk_Score
NaN       226599
0.00       52514
658.00      6127
647.00      6080
659.00      6076
Name: count, dtype: int64

--- Debt-To-Income Ratio (str), unique: 25,793
Debt-To-Income Ratio
0%         75404
-1%        33397
9999%       9939
99999%      2592
199998%     1391
Name: count, dtype: int64

--- Zip Code (str), unique: 981
Zip Code
112xx    10312
331xx  

### Observations from the sample

| Column | Observation | Decision | New name |
| --- | --- | --- | --- |
| `Amount Requested` | numeric, no missing values | keep | `amount_requested` |
| `Application Date` | text, `YYYY-MM-DD` | keep, convert to date | `application_date` |
| `Loan Title` | 61K unique values, partly free text | drop | – |
| `Risk_Score` | 23% missing, 52K zeros | keep, 0 -> missing | `risk_score` |
| `Debt-To-Income Ratio` | text with `%`, placeholders such as -1%, 9999%, 99999% | keep, convert, invalid -> missing | `dti` |
| `Zip Code` | only first 3 digits | drop (state is enough) | – |
| `State` | 51 values, very few missing | keep | `addr_state` |
| `Employment Length` | 2% missing, 76% `< 1 year` | keep, missing -> `Unknown` | `emp_length` |
| `Policy Code` | only 0 and 2 | keep for reference | `policy_code` |

Notes:
- The `dti` placeholders are not real ratios. The same rule as in `loans` is applied: values below 0 or above 100 become missing.
- 76% `< 1 year` in employment length (vs ~8% for accepted loans) may be a form default rather than real employment length. Use with care.

### Total row count
The full file is counted in chunks of 1M rows so that only one chunk is in memory at a time.

In [6]:
total_rows = 0
for chunk in pd.read_csv(REJECTED_FILE, usecols=["Application Date"], chunksize=1_000_000):
    total_rows += len(chunk)

print(f"Total rows: {total_rows:,}")

Total rows: 27,648,741


## 3. Cleaning function
All cleaning steps are wrapped in one function so the same rules are applied to every chunk of the full file.
Column names follow the `loans` table (`addr_state`, `dti`, `emp_length`) to make joins easier.

In [7]:
USE_COLUMNS = [
    "Amount Requested", "Application Date", "Risk_Score",
    "Debt-To-Income Ratio", "State", "Employment Length", "Policy Code",
]

RENAME_MAP = {
    "Amount Requested": "amount_requested",
    "Application Date": "application_date",
    "Risk_Score": "risk_score",
    "Debt-To-Income Ratio": "dti",
    "State": "addr_state",
    "Employment Length": "emp_length",
    "Policy Code": "policy_code",
}

In [8]:
def clean_rejected(chunk: pd.DataFrame) -> pd.DataFrame:
    """Clean one chunk of the rejected applications file."""
    chunk = chunk.rename(columns=RENAME_MAP)

    # dates
    chunk["application_date"] = pd.to_datetime(
        chunk["application_date"], format="%Y-%m-%d", errors="coerce"
    )

    # dti: '38.64%' -> 38.64, invalid placeholders -> missing (same rule as loans)
    chunk["dti"] = pd.to_numeric(chunk["dti"].str.rstrip("%"), errors="coerce")
    chunk.loc[(chunk["dti"] < 0) | (chunk["dti"] > 100), "dti"] = np.nan

    # risk score: 0 means no score
    chunk.loc[chunk["risk_score"] <= 0, "risk_score"] = np.nan

    # amount: 0 is not a valid request
    chunk.loc[chunk["amount_requested"] <= 0, "amount_requested"] = np.nan

    # employment length: missing -> 'Unknown' (same rule as loans)
    chunk["emp_length"] = chunk["emp_length"].fillna("Unknown")

    # drop rows without a valid date or state (needed for joins)
    chunk = chunk.dropna(subset=["application_date", "addr_state"])

    # smaller data types
    chunk["amount_requested"] = chunk["amount_requested"].astype("float32")
    chunk["risk_score"] = chunk["risk_score"].astype("float32")
    chunk["dti"] = chunk["dti"].astype("float32")
    chunk["policy_code"] = chunk["policy_code"].astype("Int8")

    return chunk

Test the function on the 1M-row sample before running it on the full file.

In [9]:
test = clean_rejected(sample[USE_COLUMNS].copy())

print(test.shape)
test.info(memory_usage="deep")

(999979, 7)
<class 'pandas.DataFrame'>
Index: 999979 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   amount_requested  999968 non-null  float32       
 1   application_date  999979 non-null  datetime64[us]
 2   risk_score        720868 non-null  float32       
 3   dti               928699 non-null  float32       
 4   addr_state        999979 non-null  str           
 5   emp_length        999979 non-null  str           
 6   policy_code       999979 non-null  Int8          
dtypes: Int8(1), datetime64[us](1), float32(3), str(2)
memory usage: 53.5 MB


In [10]:
test.describe().T

,count,mean,min,25%,50%,75%,max,std
amount_requested,"999,968.00","13,205.81",500.00,"5,000.00","10,000.00","20,000.00","1,400,000.00","12,807.92"
application_date,999979,2012-09-20 06:21:49.009089,2007-05-26 00:00:00,2011-04-06 00:00:00,2012-05-01 00:00:00,2012-12-27 00:00:00,2016-04-19 00:00:00,NaN
risk_score,"720,868.00",637.05,323.00,595.00,649.00,687.00,972.00,73.42
dti,"928,699.00",20.48,0.00,6.33,16.93,30.04,100.00,17.88
policy_code,"999,979.00",0.00,0.00,0.00,0.00,0.00,2.00,0.06


In [11]:
print(test.isna().sum())
print(test["emp_length"].value_counts().head())
print(test["application_date"].min(), "->", test["application_date"].max())

amount_requested        11
application_date         0
risk_score          279111
dti                  71280
addr_state               0
emp_length               0
policy_code              0
dtype: int64
emp_length
< 1 year     757776
5 years       54194
10+ years     42299
2 years       27274
1 year        26368
Name: count, dtype: int64
2007-05-26 00:00:00 -> 2016-04-19 00:00:00


## 4. Clean the full file in chunks
The file is read and cleaned in chunks of 1M rows, then the cleaned chunks are concatenated.

Lesson learned: the first run failed with `IntCastingNaNError` because `policy_code` has missing values
in later parts of the file that did not appear in the sample. `Int8` (nullable integer) is used instead of `int8`.

In [12]:
cleaned_chunks = []
rows_read = 0

reader = pd.read_csv(REJECTED_FILE, usecols=USE_COLUMNS, chunksize=1_000_000, low_memory=False)

for i, chunk in enumerate(reader, start=1):
    rows_read += len(chunk)
    cleaned_chunks.append(clean_rejected(chunk))
    print(f"Chunk {i:>2}: {rows_read:>12,} rows read")

rejected = pd.concat(cleaned_chunks, ignore_index=True)
del cleaned_chunks

print(f"\nRows read:     {rows_read:,}")
print(f"Rows kept:       {len(rejected):,}")
print(f"Rows dropped:    {rows_read - len(rejected):,}")

Chunk  1:    1,000,000 rows read
Chunk  2:    2,000,000 rows read
Chunk  3:    3,000,000 rows read
Chunk  4:    4,000,000 rows read
Chunk  5:    5,000,000 rows read
Chunk  6:    6,000,000 rows read
Chunk  7:    7,000,000 rows read
Chunk  8:    8,000,000 rows read
Chunk  9:    9,000,000 rows read
Chunk 10:   10,000,000 rows read
Chunk 11:   11,000,000 rows read
Chunk 12:   12,000,000 rows read
Chunk 13:   13,000,000 rows read
Chunk 14:   14,000,000 rows read
Chunk 15:   15,000,000 rows read
Chunk 16:   16,000,000 rows read
Chunk 17:   17,000,000 rows read
Chunk 18:   18,000,000 rows read
Chunk 19:   19,000,000 rows read
Chunk 20:   20,000,000 rows read
Chunk 21:   21,000,000 rows read
Chunk 22:   22,000,000 rows read
Chunk 23:   23,000,000 rows read
Chunk 24:   24,000,000 rows read
Chunk 25:   25,000,000 rows read
Chunk 26:   26,000,000 rows read
Chunk 27:   27,000,000 rows read
Chunk 28:   27,648,741 rows read

Rows read:     27,648,741
Rows kept:       27,648,719
Rows dropped:    22


`application_month` (first day of the month) is the join key to `accepted_monthly` and `dim_date`.
In `loans`, `issue_date` is already month-level, so both tables match without converting dates in every query.

In [13]:
rejected["application_month"] = (
    rejected["application_date"].dt.to_period("M").dt.to_timestamp()
)

rejected[["application_date", "application_month"]].sample(5)

,application_date,application_month
15932202,2018-11-13,2018-11-01
5844706,2018-08-21,2018-08-01
23255197,2015-08-07,2015-08-01
17967036,2018-04-29,2018-04-01
5935836,2018-08-24,2018-08-01


Repeated text columns are converted to `category` **after** concatenation.
Converting inside the function would give each chunk different categories, and `concat` would turn them back into text.

In [14]:
rejected["addr_state"] = rejected["addr_state"].astype("category")
rejected["emp_length"] = rejected["emp_length"].astype("category")

rejected.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 27648719 entries, 0 to 27648718
Data columns (total 8 columns):
 #   Column             Dtype         
---  ------             -----         
 0   amount_requested   float32       
 1   application_date   datetime64[us]
 2   risk_score         float32       
 3   dti                float32       
 4   addr_state         category      
 5   emp_length         category      
 6   policy_code        Int8          
 7   application_month  datetime64[us]
dtypes: Int8(1), category(2), datetime64[us](2), float32(3)
memory usage: 843.8 MB


## 5. Checks

In [15]:
print(rejected["application_date"].min(), "->", rejected["application_date"].max())
print()
print(rejected["policy_code"].value_counts())
print()
print(rejected.isna().sum())

2007-05-26 00:00:00 -> 2018-12-31 00:00:00

policy_code
0    27559672
2       88129
Name: count, dtype: Int64

amount_requested         1288
application_date            0
risk_score           18584381
dti                   2011012
addr_state                  0
emp_length                  0
policy_code               918
application_month           0
dtype: int64


In [16]:
yearly = rejected.groupby(rejected["application_date"].dt.year).agg(
    applications=("amount_requested", "size"),
    median_amount=("amount_requested", "median"),
    risk_score_missing_pct=("risk_score", lambda s: s.isna().mean() * 100),
    risk_score_min=("risk_score", "min"),
    risk_score_median=("risk_score", "median"),
    risk_score_max=("risk_score", "max"),
    dti_median=("dti", "median"),
)
yearly

,applications,median_amount,risk_score_missing_pct,risk_score_min,risk_score_median,risk_score_max,dti_median
application_date,,,,,,,
2007,5274,"6,000.00",6.20,363.00,576.00,811.00,14.12
2008,25575,"7,000.00",13.81,363.00,613.00,821.00,14.67
2009,56991,"8,500.00",16.76,364.00,638.00,828.00,15.22
2010,112561,"9,000.00",15.55,373.00,654.00,849.00,16.70
2011,217792,"10,000.00",10.30,367.00,650.00,832.00,15.28
2012,337277,"10,000.00",6.85,385.00,651.00,850.00,17.12
2013,760942,"10,000.00",7.63,390.00,649.00,990.00,17.79
2014,1933700,"9,000.00",13.16,300.00,638.00,990.00,17.96
2015,2859379,"10,000.00",82.15,300.00,641.00,990.00,19.20


### Findings by year
- Rejected applications grew from ~5K (2007) to ~9.5M (2018).
- Rough approval rate for fully mature years: 2010 ≈ 10%, 2013 ≈ 15% of all applications.
- `risk_score` is **not usable** for analysis: the scale changes from 2013 (max 990 instead of 850)
  and 82–93% of values are missing in 2015, 2016 and 2018. The column is kept but not used in the main analysis.
- Median requested amount is stable (~$10K), while median DTI of rejected applicants rose from 14% to 21%.

## 6. Save

In [22]:
PROCESSED_FILE = Path("../data/processed/rejected_clean.parquet")

rejected.to_parquet(PROCESSED_FILE, index=False)
print(f"Saved: {PROCESSED_FILE} ({PROCESSED_FILE.stat().st_size / 1e6:.0f} MB)")

check = pd.read_parquet(PROCESSED_FILE)
print(check.shape)
print(check.dtypes)

Saved: ..\data\processed\rejected_clean.parquet (129 MB)
(27648719, 8)
amount_requested            float32
application_date     datetime64[us]
risk_score                  float32
dti                         float32
addr_state                 category
emp_length                 category
policy_code                    Int8
application_month    datetime64[us]
dtype: object


## 7. Load to PostgreSQL
27.6M rows are loaded with PostgreSQL `COPY` (bulk load), which is much faster than row-by-row inserts (`to_sql`).
The load runs in one transaction: if any chunk fails, nothing is saved.

In [ ]:
load_dotenv("../.env")
engine = create_engine(os.getenv("DATABASE_URL"))

# Reload from parquet if the notebook was restarted
rejected = pd.read_parquet("../data/processed/rejected_clean.parquet")

COPY_COLUMNS = [
    "application_date", "application_month", "amount_requested",
    "risk_score", "dti", "addr_state", "emp_length", "policy_code",
]
COPY_SQL = (
    f"COPY rejected_applications ({', '.join(COPY_COLUMNS)}) "
    "FROM STDIN WITH (FORMAT csv)"
)
print(COPY_SQL)
print(f"{len(rejected):,} rows to load")

COPY rejected_applications (application_date, application_month, amount_requested, risk_score, dti, addr_state, emp_length, policy_code) FROM STDIN WITH (FORMAT csv)
27,648,719 rows to load


In [25]:
CHUNK_SIZE = 1_000_000
start = time.time()

connection = engine.raw_connection()
try:
    with connection.cursor() as cursor:
        for i in range(0, len(rejected), CHUNK_SIZE):
            chunk = rejected.iloc[i : i + CHUNK_SIZE][COPY_COLUMNS]

            buffer = io.StringIO()
            chunk.to_csv(buffer, index=False, header=False, date_format="%Y-%m-%d")
            buffer.seek(0)

            cursor.copy_expert(COPY_SQL, buffer)
            print(f"{i + len(chunk):>12,} rows sent   ({time.time() - start:.0f} s)")

    connection.commit()
    print("Committed.")
except Exception:
    connection.rollback()
    raise
finally:
    connection.close()

   1,000,000 rows sent   (16 s)
   2,000,000 rows sent   (33 s)
   3,000,000 rows sent   (49 s)
   4,000,000 rows sent   (66 s)
   5,000,000 rows sent   (83 s)
   6,000,000 rows sent   (100 s)
   7,000,000 rows sent   (117 s)
   8,000,000 rows sent   (135 s)
   9,000,000 rows sent   (153 s)
  10,000,000 rows sent   (170 s)
  11,000,000 rows sent   (189 s)
  12,000,000 rows sent   (205 s)
  13,000,000 rows sent   (223 s)
  14,000,000 rows sent   (241 s)
  15,000,000 rows sent   (259 s)
  16,000,000 rows sent   (276 s)
  17,000,000 rows sent   (294 s)
  18,000,000 rows sent   (311 s)
  19,000,000 rows sent   (328 s)
  20,000,000 rows sent   (345 s)
  21,000,000 rows sent   (364 s)
  22,000,000 rows sent   (383 s)
  23,000,000 rows sent   (399 s)
  24,000,000 rows sent   (416 s)
  25,000,000 rows sent   (434 s)
  26,000,000 rows sent   (451 s)
  27,000,000 rows sent   (468 s)
  27,648,719 rows sent   (478 s)
Committed.


### Index and verification
The index on `(application_month, addr_state)` is created **after** loading, because building it on a full table
is faster than updating it for every inserted row. `IF NOT EXISTS` makes the cell safe to re-run.

In [26]:
with engine.begin() as conn:
    conn.execute(text(
        "CREATE INDEX IF NOT EXISTS idx_rejected_month_state "
        "ON rejected_applications (application_month, addr_state)"
    ))
    conn.execute(text("ANALYZE rejected_applications"))

with engine.connect() as conn:
    db_rows = conn.execute(text("SELECT COUNT(*) FROM rejected_applications")).scalar()
    indexes = conn.execute(text(
        "SELECT indexname FROM pg_indexes WHERE tablename = 'rejected_applications'"
    )).scalars().all()

print(f"Rows in parquet:  {len(rejected):,}")
print(f"Rows in database: {db_rows:,}")
print(f"Indexes:          {indexes}")

Rows in parquet:  27,648,719
Rows in database: 27,648,719
Indexes:          ['rejected_applications_pkey', 'idx_rejected_month_state']


## Summary

| Step | Rows |
| --- | --- |
| Raw file | 27,648,741 |
| Rows without state dropped | 27,648,719 |
| Loaded to PostgreSQL (`rejected_applications`) | 27,648,719 |

**Output:** `data/processed/rejected_clean.parquet` (8 columns, ~129 MB) and the `rejected_applications` table.

**Limitations:**
- `risk_score` changes scale over time and is mostly missing after 2014; not used in the analysis.
- Rejected applications use the application date, accepted loans use the issue date. At month level the gap is small.